# One-Galaxy Kinematic Lensing Inference

**Showcase notebook.** This is the main end-to-end demonstration of KINO: starting
from a single intrinsic disc galaxy, we apply a *known* shear, simulate what a
chosen instrument would observe, and then run inference to recover that shear
back out of the mock data. Because the input shear is known, this is an
injection-recovery test — the simplest possible check that the whole pipeline
(forward model + instrument + inference) is self-consistent.

Pipeline:

```
intrinsic disc (spectrum + Vlos) -> apply shear (g1, g2) -> instrument backend -> mock data
                                                                                        |
                              nautilus posterior on (g1, g2)  <-------------------------
```

Two instrument backends are supported on the same forward model:

- **IFU** — mock data is an **image** + a **datacube** (spatially and
  spectrally resolved).
- **slit/grism spectrograph** — mock data is an **image** + a **measured
  (dispersed) spectrum** (spatially unresolved or partially resolved along the
  slit).

> **Status:** this notebook is a scaffold. The narrative and the intended API
> calls are laid out below, but most `kino` modules are still empty
> (Phases 1-5 of the roadmap in the top-level `README.md`). Code cells are
> written against the target API from `README.md`'s Quickstart and will raise
> `NotImplementedError` / `ImportError` until each phase lands — fill them in
> in place rather than restructuring the notebook.

## 0. Setup

In [ ]:
import numpy as np

import kino
from kino.galaxy import DiscGalaxy
from kino.inference import fit_shear
from kino.instruments import IFU, SlitSpectrograph
from kino.lensing import apply_shear

rng = np.random.default_rng(seed=0)  # explicit, seeded RNG -- see CLAUDE.md conventions
print(kino.__version__)

## 1. Intrinsic disc galaxy

Define the unlensed galaxy: an emission-line spectrum (for the line-of-sight
velocity field) plus the disc geometry (inclination, position angle,
effective radius, systemic velocity, rotation curve). Positions in arcsec,
velocities in km/s, angles in degrees -- see `CLAUDE.md` for the full unit and
sign conventions.

In [ ]:
gal = DiscGalaxy(
    r_e=0.5,  # effective radius, arcsec
    inc=60,  # inclination, degrees (0 = face-on, 90 = edge-on)
    pa=35,  # position angle, degrees East of North
    v_flat=180,  # asymptotic rotation velocity, km/s
    z=0.8,  # redshift
    rng=rng,
)
gal

## 2. Apply a known shear

This is the ground truth for the injection-recovery test below: we shear the
disc by a chosen `(g1, g2)` and later check that inference recovers it. The
shear is applied jointly to the surface brightness **and** the velocity
field -- shearing only the morphology is a classic bug (see `CLAUDE.md`,
"Things to be careful about").

In [ ]:
g1_true, g2_true = 0.03, -0.01
scene = apply_shear(gal, g1=g1_true, g2=g2_true)
scene

## 3. Choose an instrument

Instrument behaviour (PSF, LSF, spatial/spectral sampling, throughput, noise)
lives entirely behind the `Instrument.observe(scene)` API and a YAML config --
never in the galaxy or lensing code. Swap the config (and the class) to
switch between the IFU and slit/grism backends without touching the steps
above.

In [ ]:
ifu = IFU.from_config("../config/gmtifs.yaml")
ifu

## 4a. Simulate: IFU backend

Produces a simulated **image** and a simulated **datacube** (spatially and
spectrally resolved `Vlos`, flux, and dispersion).

In [ ]:
ifu_data = ifu.observe(scene, exp_time=3 * 3600, rng=rng)  # exp_time in seconds
image_ifu, cube = ifu_data.image, ifu_data.cube
image_ifu.shape, cube.shape

## 4b. Simulate: slit/grism backend

Produces a simulated **image** (for slit placement / acquisition) and a
simulated **measured spectrum** (the dispersed, slit-integrated signal,
including slit losses and misalignment).

In [ ]:
slit = SlitSpectrograph.from_config("../config/gmacs.yaml")
slit_data = slit.observe(scene, exp_time=3 * 3600, rng=rng)
image_slit, spectrum = slit_data.image, slit_data.spectrum
image_slit.shape, spectrum.shape

## 5. Visualize the mock data

Sanity-check the forward model before running inference: flux map, recovered
`Vlos` map (IFU) or dispersed spectrum (slit/grism), and the imaging.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_ifu)
axes[0].set_title("image")
axes[1].imshow(cube.sum(axis=0))
axes[1].set_title("cube, summed over wavelength")
axes[2].imshow(cube.moment(1))  # e.g. flux-weighted Vlos map
axes[2].set_title("Vlos moment map")
fig.tight_layout()

## 6. Run inference: recover the (g1, g2) posterior

The likelihood compares the model (galaxy + shear + instrument, forward-run
for trial parameters) against the mock data; priors include the Tully-Fisher
relation on rotation velocity. Posterior sampling uses **nautilus**
(neural-network-boosted importance nested sampling) behind the common
`kino.inference` sampler API.

In [ ]:
posterior = fit_shear(ifu_data, sampler="nautilus")
posterior.summary()  # g1, g2 with credible intervals

## 7. Injection-recovery check

Compare the recovered posterior to the injected truth `(g1_true, g2_true)`
from step 2 -- the core validation of the whole pipeline.

In [ ]:
posterior.corner_plot(truth={"g1": g1_true, "g2": g2_true})

## Next steps

- Repeat with the slit/grism mock (`slit_data`) and compare IFU vs.
  slit/grism precision on the same injected shear (Phase 6 of the roadmap).
- Run an ensemble of galaxies at fixed shear to characterise residual shape
  noise and multiplicative/additive (m, c) bias (`kino.analysis`).
- See `examples/simulate_one_galaxy.py` and `examples/run_inference.py` for
  the same pipeline as reproducible, config-driven CLI scripts.